# Building an Agent, Step by Step
**LangChain v1 + LangGraph + LangSmith**

Each stage fixes one limitation of the stage before it.

## Stage 0: Setup

In [2]:
# pip install -U langchain langgraph langchain-anthropic langchain-tavily python-dotenv numpy
from dotenv import load_dotenv
load_dotenv(override=True)   # loads ANTHROPIC_API_KEY and LANGSMITH_* settings from .env

MODEL = "anthropic:claude-haiku-5-5"                  # group key model (claude-sonnet-5-5 only when told)

## Stage 1: Talk to an LLM
Type `quit` to stop. Tell it your name, then ask it back.

In [4]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(MODEL)

while True:
    user = input("You: ")
    if user == "quit":
        break
    print("You:", user)

    reply = llm.invoke(user)   # send ONLY this one message
    print("AI:", reply.content)

You: how you doing
AI: I'm doing well, thanks for asking! How about you? How's your day going?
You: 


AnthropicInvalidRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.0: user messages must have non-empty content'}, 'request_id': 'req_011ChKi1a1WutRDCEpUyzd4L'}

## Stage 2: Short-term memory
Memory is just *sending the history back every time*. First by hand:

In [15]:
history = []

while True:
    user = input("You: ")
    if user == "quit":
        break
    print("You:", user)

    # 1. add what the user said to the history
    history.append({"role": "user", "content": user})

    # 2. send the WHOLE history, not just the last message
    reply = llm.invoke(history)

    # 3. add the AI's answer to the history too
    history.append({"role": "assistant", "content": reply.content})
    print("AI:", reply.content)

In [ ]:
history

Now let the framework handle it. A **checkpointer** saves the conversation per `thread_id`, so we never touch the list.

In [16]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

def ask(agent, text, thread="1"):
    # The new message we send to the agent
    new_message = {"role": "user", "content": text}

    # Which conversation this belongs to. The checkpointer saves history per thread_id.
    config = {"configurable": {"thread_id": thread}}

    result = agent.invoke({"messages": [new_message]}, config)

    # result["messages"] is the whole conversation; the last one is the agent's reply
    last_message = result["messages"][-1]
    return last_message.content

def chat(agent, thread):
    while True:
        user = input("You: ")
        if user == "quit":
            break
        print("You:", user)
        assistant_response = ask(agent, user, thread)
        print("AI:", assistant_response)

agent = create_agent(MODEL, checkpointer=InMemorySaver())
chat(agent, thread="1") 

In [ ]:
chat(agent, thread="1")    

## Stage 3: Tools — from chatbot to agent
It can *answer*, but it can't *do* anything. Tools are plain functions; the **docstring** is what the model reads.

In [17]:
from langchain.tools import tool

# A fake timetable. In a real app this would be a database or an API.
TIMETABLE = {
    "CS101": "Mon/Wed 10:00, Room B2",
    "AI301": "Tue/Thu 14:00, Lab 4",
}

@tool
def check_timetable(course_code: str) -> str:
    """Get the class schedule for a course code like CS101."""
    if course_code in TIMETABLE:
        return TIMETABLE[course_code]
    return "Not found"

@tool
def book_study_room(room: str, time: str) -> str:
    """Book a library study room at a given time."""
    return f"✅ Booked {room} at {time}"

tools = [check_timetable, book_study_room]
agent = create_agent(MODEL, tools=tools, checkpointer=InMemorySaver())
ask(agent, "When is AI301? Book me study room 3 an hour before it.")

'AI301 is scheduled for Tuesdays and Thursdays at 14:00. I have successfully booked study room 3 for you at 13:00 on October 31, 2023.'

👉 Open LangSmith and show the trace: model → tool call → tool result → model.

## Stage 4: Web search — what's happening *today*?
It can act, but it only knows its training data. **Tavily** is a *prebuilt* tool: nothing to write, just add it to the list.

In [19]:
from langchain_tavily import TavilySearch

# A prebuilt tool: someone already wrote it. Needs TAVILY_API_KEY in .env (free key at tavily.com)
web_search = TavilySearch(max_results=3)

tools = [check_timetable, book_study_room, web_search]
agent = create_agent(MODEL, tools=tools, checkpointer=InMemorySaver())

# print() each answer: a notebook cell only shows its LAST value on its own
print(ask(agent, "What are people's thoughts about GTA 6"))
print()
print(ask(agent, "What's the weather in London right now?"))
print()
print(ask(agent, "When is AI301?"))   # the timetable tool, not the web

1. **IGN's Review**: GTA 6 is described as "one of the most impressive games ever made," thanks to its stunning graphics and realistic world. The game features enhanced car handling reminiscent of GTA 4 and maintains a focus on significant heist missions. [Read more here](https://www.ign.com/articles/grand-theft-auto-vi-review).

2. **Eurogamer's Review**: The review emphasizes the game’s vastness combined with intricate details. It presents a mix of themes—big ideas alongside personal stories—while offering a blend of playful and serious content. However, it notes some tonal inconsistencies in character development. [Read more here](https://www.eurogamer.net/grand-theft-auto-6-review).

3. **GameSpot's Review**: This review highlights the game’s deliberate animation style that adds weight to interactions in the game world, enhancing immersion. Although it praised the game overall, it mentions some challenges with car physics, particularly at high speeds. [Read more here](https://www.g

## Stage 5: RAG — give it *our* knowledge
The web doesn't know *our* university policy. Load the handbook, chunk it, embed it, and give the agent a **search tool**: just another tool.

In [21]:
from pathlib import Path
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
import sys; sys.path.insert(0, "..")
from app.local_embeddings import LocalEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.tools import tool

# 1. LOAD: read every markdown file in docs/ into a Document (text + where it came from)
docs = []
for path in Path("../docs").glob("*.md"):
    text = path.read_text()
    docs.append(Document(text, metadata={"source": path.name}))

# 2. CHUNK: split long documents into small overlapping pieces
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(docs)

# 3. EMBED + STORE: turn each chunk into numbers (an embedding) and keep them searchable
embeddings = LocalEmbeddings()
vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)

print(f"Loaded {len(docs)} document(s), split into {len(chunks)} chunks")

# 4. SEARCH TOOL: find the 3 chunks closest in meaning to the question
@tool
def search_handbook(query: str) -> str:
    """Search the university handbook for policies: exams, attendance, grading, fees."""
    results = vector_store.similarity_search(query, k=3)

    pieces = []
    for doc in results:
        pieces.append(doc.page_content)

    return "\n\n".join(pieces)   # glue the 3 chunks together with blank lines between them

Loaded 1 document(s), split into 1 chunks


In [22]:
SYSTEM_PROMPT = ("You are a university assistant. "
                 "Use the handbook for policy questions and web search for anything current.")

tools = [search_handbook, check_timetable, book_study_room, web_search]
agent = create_agent(MODEL, tools=tools, checkpointer=InMemorySaver(), system_prompt=SYSTEM_PROMPT)

ask(agent, "check my timetable for AI301")   # the handbook, not the web

'Your timetable for AI301 is as follows:\n\n- **Lectures:** Tuesday and Thursday at 14:00\n- **Lab:** Lab 4 \n\nIf you have any further questions or need additional information, feel free to ask!'

## Stage 6: Human-in-the-loop
Actions with consequences should ask first. One line of middleware.

In [23]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

# Pause before book_study_room runs. Other tools still run freely.
approval = HumanInTheLoopMiddleware(interrupt_on={"book_study_room": True})

agent = create_agent(MODEL, tools=tools, checkpointer=InMemorySaver(), middleware=[approval])

config = {"configurable": {"thread_id": "hitl"}}
new_message = {"role": "user", "content": "Book room 3 at 5pm"}
result = agent.invoke({"messages": [new_message]}, config)

result["__interrupt__"]                    # ⏸️ paused, waiting for a human

[Interrupt(value={'action_requests': [{'name': 'book_study_room', 'args': {'room': '3', 'time': '5pm'}, 'description': "Tool execution requires approval\n\nTool: book_study_room\nArgs: {'room': '3', 'time': '5pm'}"}], 'review_configs': [{'action_name': 'book_study_room', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='10fa22b60f3a4f15a773597fabaa6b42', response_schema=None)]

In [24]:
# The human's decision: "approve" (or "reject")
decision = {"decisions": [{"type": "approve"}]}

# Command(resume=...) continues the paused run on the same thread
result = agent.invoke(Command(resume=decision), config)

result["messages"][-1].content

'Room 3 has been successfully booked for you at 5 PM.'